# Camera-ready LRao — definitive run (all scenes, 5 seeds, validation early stopping)
**Upload-and-run on a GPU runtime** (CPU also works — minutes per seed).

Trains LRao on the three spatial scenes (**pavia4, sandiego, sandiego2**),
**5 seeds each**, and scores the full amplitude sweep (θ = 0.03 … 0.95, the
same grid and planted test sets as the archived detector sweep — labels match
npz-for-npz).

**Recipe = the LRao paper's prescribed usage**: a **20% validation split**
with **early stopping on the validation LFI cost** (stop after 20 epochs
without a new validation minimum; keep the best-validation model). The model,
its robust median/MAD whitening, its scoring reference set and its CFAR
thresholds use only the remaining 80% — the sample-budget tradeoff noted in
the paper. Optimizer knobs follow the recovered July-7 recipe (Adam 5e-4 /
wd 5e-5, batch 2048, `sigma_cutoff=1e-22` in the loss, grad-clip 1.0,
≤1000 epochs) with **hidden_dims=[128] — the same one-hidden-layer
architecture as DART**.

**Why validation stopping**: on San Diego II the training LFI cost decreases
monotonically while detection oscillates and collapses (verified from the
every-10-epoch checkpoints), so no train-cost rule can select a good model.

**Archived per run**: a checkpoint every 10 epochs + best/final weights +
train AND validation loss curves; raw train-subset and test scores per
(scene, seed, θ); per-seed metrics (AUC, pAUC, Pd@0.05, Pd_cfar, Pfa;
per-class Pfa on Pavia). Every cell is resumable.

At the end: one zip (`lrao_val.zip`) downloads automatically.


In [ ]:
!git clone -b tsp-repro --depth 1 https://github.com/michaelpiro/final-paper-experiment.git repo
%cd repo
import sys, os, torch
sys.path.insert(0, '.')
import tsp_repro
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
for p in ('colab_deep/data/pavia-u.mat', 'tsp_repro/data/Sandiego.mat',
          'tsp_repro/data/Sandiego2.mat', 'tsp_repro/data/sandiego_regions.json',
          'tsp_repro/data/sandiego2_regions.json', 'tsp_repro/configs/spatial.yaml'):
    assert os.path.exists(p), f'missing {p}'
print('all bundled datasets present')

In [ ]:
from tsp_repro import lrao_camera_ready as LC
print('recipe:'); [print(f'  {k}: {v}') for k, v in LC.RECIPE.items()]
OUT = 'results/lrao_val'

## Train + sweep, one cell per scene (each resumable)

In [ ]:
LC.run_scene('pavia4', out_root=OUT, device=DEVICE)

In [ ]:
LC.run_scene('sandiego', out_root=OUT, device=DEVICE)

In [ ]:
LC.run_scene('sandiego2', out_root=OUT, device=DEVICE)

## Summary — AUC grid per scene + the Table row at θ=0.15

In [ ]:
LC.summarize(OUT)

## Zip + download
`ckpt/` (every-10-epoch checkpoints, best/final, histories),
`scores/` (raw train/test scores incl. labels), `metrics__*.json`.

In [ ]:
LC.make_zip(OUT, 'lrao_val.zip')